In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
RESULTS_DIR = Path('../results/universality')

with open(RESULTS_DIR / 'universality_summary.json') as f:
    summary = json.load(f)

summary

In [ ]:
d_sae = summary['d_sae']
n_after_scanner = d_sae - summary['n_scanner_flagged']

stages = [
    ('Scanner-safe',       n_after_scanner),
    ('Frequency (L1)',     summary['n_level1_candidates']),
    ('Semantic (L2)',      summary['n_level2_confirmed']),
    ('Post-split',         summary['n_after_split']),
    ('Universal',          summary['n_universal']),
]

labels = [s[0] for s in stages]
values = np.array([s[1] for s in stages], dtype=float)
total  = d_sae

pct_initial = values / total * 100

x = np.arange(len(values))
upper =  values / 2
lower = -values / 2

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')

colors = ['#4A90D9', '#5B9BDE', '#6FA8E3', '#88B8EA', '#A6CBF1']
for i in range(len(x) - 1):
    ax.fill_between(
        [x[i], x[i+1]],
        [lower[i], lower[i+1]],
        [upper[i], upper[i+1]],
        color=colors[i],
        linewidth=0,
    )

# separators between stages
for xi in x:
    ax.axvline(xi, color='#E0E0E0', linewidth=1, zorder=0)

# stage labels on top
y_top = upper.max() * 1.15
for xi, lbl in zip(x, labels):
    ax.text(xi, y_top, lbl, ha='center', va='bottom',
            fontsize=12, color='#8A8A8A')

# percentages + counts below
y_pct  = -upper.max() * 1.18
y_cnt  = -upper.max() * 1.30
for xi, p, v in zip(x, pct_initial, values):
    ax.text(xi, y_pct, f'{p:.2f} %', ha='center', va='top',
            fontsize=13, color='#8A8A8A')
    ax.text(xi, y_cnt, f'{int(v):,}', ha='center', va='top',
            fontsize=11, color='#B0B0B0')

# big total label on the left
ax.text(-0.9, 0, f'{int(total):,}', ha='left', va='center',
        fontsize=28, fontweight='bold', color='#202020')
ax.text(-0.9, -upper.max() * 0.18, 'Features', ha='left', va='center',
        fontsize=14, color='#9A9A9A')

ax.set_xlim(-1.0, len(x) - 0.5)
ax.set_ylim(-upper.max() * 1.45, upper.max() * 1.35)
ax.set_axis_off()

plt.tight_layout()

out_path = RESULTS_DIR / 'universal_feature_funnel.png'
fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved to {out_path}')
plt.show()